# Gradio demo: OCR (fancy)

> Be sure to select **Runtime > Change runtime type** and pick anything other than CPU!

We're comparing a fast traditional OCR engine with a small vision-language OCR model: [RapidOCR](https://github.com/RapidAI/RapidOCR) and [Nanonets-OCR2-1.5B-exp](https://huggingface.co/nanonets/Nanonets-OCR2-1.5B-exp).

In [ ]:
from functools import lru_cache
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory

import fitz
import gradio as gr
from rapidocr import RapidOCR


DPI = 200
NANONETS_MODEL_ID = "nanonets/Nanonets-OCR2-1.5B-exp"
NANONETS_MAX_NEW_TOKENS = 4096
NANONETS_PROMPT = """Extract the text from the above document as if you were reading it naturally. Return tables in HTML format. Return equations in LaTeX representation. Use markdown where it helps preserve structure. Prefer using ☐ and ☑ for checkboxes."""
rapidocr = RapidOCR()


def render_pages(pdf_path, pages, image_dir):
    zoom = DPI / 72
    matrix = fitz.Matrix(zoom, zoom)
    image_paths = []

    with fitz.open(pdf_path) as pdf:
        for page_number in pages:
            page = pdf.load_page(page_number - 1)
            pixmap = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = image_dir / f"page-{page_number}.png"
            pixmap.save(image_path)
            image_paths.append(image_path)

    return image_paths


def preview_page(pdf_path, page_number):
    if not pdf_path or page_number is None:
        return None

    preview_file = NamedTemporaryFile(delete=False, suffix=".png")
    preview_file.close()

    with fitz.open(pdf_path) as pdf:
        page = pdf.load_page(int(page_number) - 1)
        pixmap = page.get_pixmap(matrix=fitz.Matrix(DPI / 72, DPI / 72), alpha=False)
        pixmap.save(preview_file.name)

    return preview_file.name


def load_pdf(pdf_path, start_page):
    if not pdf_path:
        return None, gr.update(value=1)

    with fitz.open(pdf_path) as pdf:
        page_count = pdf.page_count

    # Limit to 2 pages
    page_count = min(page_count, 2) 
    return preview_page(pdf_path, start_page), gr.update(value=page_count)


def run_rapidocr(image_paths):
    texts = []

    for image_path in image_paths:
        result = rapidocr(str(image_path))
        texts.append("\n".join(result.txts or []).strip())

    return texts


@lru_cache
def nanonets_ocr_model():
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor

    if torch.cuda.is_available():
        torch_dtype = torch.float16
        device_map = "auto"
    else:
        torch_dtype = torch.float32
        device_map = None

    processor = AutoProcessor.from_pretrained(NANONETS_MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        NANONETS_MODEL_ID,
        torch_dtype=torch_dtype,
        device_map=device_map,
    )
    if device_map is None:
        device = "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
        model = model.to(device)
    model.eval()
    return processor, model


def run_nanonets_ocr(image_paths):
    import torch
    from PIL import Image

    processor, model = nanonets_ocr_model()
    texts = []

    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        messages = [
            {"role": "system", "content": "You are a helpful OCR assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{image_path}"},
                    {"type": "text", "text": NANONETS_PROMPT},
                ],
            }
        ]
        prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[prompt],
            images=[image],
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=NANONETS_MAX_NEW_TOKENS,
                do_sample=False,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
        text = processor.decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        texts.append(text.strip())

    return texts


def ocr_pdf(pdf_file, start_page, end_page, engine):
    pages = list(range(int(start_page), int(end_page) + 1))

    with TemporaryDirectory() as image_dir:
        image_paths = render_pages(pdf_file, pages, Path(image_dir))
        if engine == "rapidocr":
            page_texts = run_rapidocr(image_paths)
        else:
            page_texts = run_nanonets_ocr(image_paths)

    return "\n\n".join(
        f"## Page {page_number}\n\n{text}"
        for page_number, text in zip(pages, page_texts)
    )


with gr.Blocks(title="PDF OCR demo") as demo:
    gr.Markdown("# PDF OCR demo")

    with gr.Row():
        with gr.Column():
            pdf = gr.File(label="PDF", file_types=[".pdf"], type="filepath")

            with gr.Row():
                start_page = gr.Number(label="Start page", value=1, precision=0)
                end_page = gr.Number(label="End page", value=1, precision=0)

            engine = gr.Radio(
                label="OCR engine",
                choices=["rapidocr", "nanonets_ocr2"],
                value="rapidocr",
            )

            button = gr.Button("OCR pages", variant="primary")
            preview = gr.Image(
                label="Page preview",
                type="filepath",
                height=520,
                interactive=False,
            )

        with gr.Column():
            output = gr.Textbox(label="OCR text", lines=24)

    pdf.change(load_pdf, [pdf, start_page], [preview, end_page])
    start_page.change(preview_page, [pdf, start_page], preview)
    button.click(ocr_pdf, [pdf, start_page, end_page, engine], output)


In [ ]:
import os
os.environ['GRADIO_DEBUG'] = '1'

demo.launch()
